# Stability and Qualitative Analysis of ODEs
# Stabilité et Analyse Qualitative des EDO

**AIMS Master's Programme — ODE Course**

We study the stability of equilibrium points (points d'équilibre), linearisation, classification of equilibria, Lyapunov stability concepts, and bifurcation analysis. These qualitative tools let us understand the long-term behaviour of solutions without computing them explicitly.

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from numpy.linalg import eig
from scipy.optimize import fsolve

plt.rcParams.update({'figure.figsize': (10, 6), 'font.size': 12})

## 1. Equilibrium Points (Points d'équilibre / Points fixes)

For the autonomous system $\mathbf{x}' = \mathbf{F}(\mathbf{x})$, an **equilibrium point** $\mathbf{x}^*$ satisfies $\mathbf{F}(\mathbf{x}^*) = \mathbf{0}$.

### Example: Logistic growth with harvesting

$$\frac{dN}{dt} = rN\left(1 - \frac{N}{K}\right) - H$$

where $r$ is the growth rate, $K$ the carrying capacity (capacité de charge), and $H$ the constant harvest rate. Setting the right-hand side to zero gives a quadratic in $N$:

$$rN\left(1 - \frac{N}{K}\right) = H \implies N^2 - KN + \frac{KH}{r} = 0$$

In [ ]:
# Logistic growth with constant harvesting
r, K = 1.0, 100.0

def logistic_harvest(t, N, H):
    return r * N * (1 - N/K) - H

# Find and plot equilibria for different harvest rates
N_range = np.linspace(0, 120, 300)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: dN/dt vs N for different H
for H_val in [0, 10, 20, 25, 30]:
    dNdt = r * N_range * (1 - N_range/K) - H_val
    ax1.plot(N_range, dNdt, linewidth=2, label=f'H={H_val}')

ax1.axhline(y=0, color='k', linewidth=0.5)
ax1.set_xlabel('Population $N$'); ax1.set_ylabel('$dN/dt$')
ax1.set_title('Growth rate vs population for different harvest rates')
ax1.legend(); ax1.grid(True, alpha=0.3)

# Right: time evolution
H_val = 20
t_span = (0, 30)
t_eval = np.linspace(*t_span, 300)
for N0 in [10, 30, 50, 70, 90, 110]:
    sol = solve_ivp(logistic_harvest, t_span, [N0], args=(H_val,),
                    t_eval=t_eval, events=lambda t, y: y[0])  # stop at N=0
    ax2.plot(sol.t, sol.y[0], linewidth=1.5)

# Equilibria for H=20: N = (K ± sqrt(K^2 - 4KH/r)) / 2
disc = K**2 - 4*K*H_val/r
N_star_1 = (K - np.sqrt(disc)) / 2
N_star_2 = (K + np.sqrt(disc)) / 2
ax2.axhline(y=N_star_1, color='red', linestyle='--', label=f'$N^*_1 = {N_star_1:.1f}$ (unstable)')
ax2.axhline(y=N_star_2, color='green', linestyle='--', label=f'$N^*_2 = {N_star_2:.1f}$ (stable)')
ax2.set_xlabel('Time'); ax2.set_ylabel('Population $N$')
ax2.set_title(f'Logistic with harvesting $H={H_val}$')
ax2.legend(fontsize=9); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
# Figure: Left — dN/dt curves; equilibria are where curves cross zero.
# Right — trajectories converge to stable equilibrium or collapse.

## 2. Linearisation Around Equilibria (Linéarisation)

For a 2D system $\mathbf{x}' = \mathbf{F}(\mathbf{x})$ with equilibrium $\mathbf{x}^*$, the **linearised system** is:

$$\mathbf{u}' = J(\mathbf{x}^*)\,\mathbf{u}, \quad \text{where } \mathbf{u} = \mathbf{x} - \mathbf{x}^*$$

and $J$ is the **Jacobian matrix** (matrice jacobienne):

$$J = \begin{pmatrix} \partial F_1/\partial x_1 & \partial F_1/\partial x_2 \\ \partial F_2/\partial x_1 & \partial F_2/\partial x_2 \end{pmatrix}\bigg|_{\mathbf{x}^*}$$

The eigenvalues of $J$ determine local stability (by the **Hartman-Grobman theorem**, provided no eigenvalue has zero real part).

### Example: Predator-Prey with logistic prey

In [ ]:
# Modified Lotka-Volterra with logistic prey growth
# dx/dt = r*x*(1 - x/K) - a*x*y
# dy/dt = b*x*y - m*y

r_pp, K_pp, a_pp, b_pp, m_pp = 1.0, 100.0, 0.02, 0.01, 0.3

def predator_prey(t, z):
    x, y = z
    return [r_pp*x*(1 - x/K_pp) - a_pp*x*y,
            b_pp*x*y - m_pp*y]

# Equilibria:
# (0, 0), (K, 0), and (x*, y*) where x* = m/b, y* = (r/a)*(1 - x*/K)
x_star = m_pp / b_pp
y_star = (r_pp / a_pp) * (1 - x_star / K_pp)
equilibria = [(0, 0), (K_pp, 0), (x_star, y_star)]

print("Equilibrium points and their stability:")
print("="*60)

for eq in equilibria:
    x0, y0 = eq
    # Jacobian
    J = np.array([
        [r_pp*(1 - 2*x0/K_pp) - a_pp*y0, -a_pp*x0],
        [b_pp*y0, b_pp*x0 - m_pp]
    ])
    eigenvalues = np.linalg.eig(J)[0]
    
    # Classify
    re_parts = eigenvalues.real
    im_parts = eigenvalues.imag
    if all(re_parts < 0):
        if any(im_parts != 0):
            stability = "Stable spiral (foyer stable)"
        else:
            stability = "Stable node (noeud stable)"
    elif all(re_parts > 0):
        stability = "Unstable node/spiral"
    elif re_parts[0] * re_parts[1] < 0:
        stability = "Saddle point (col)"
    else:
        stability = "Center or non-hyperbolic"
    
    print(f"\nEquilibrium: ({x0:.1f}, {y0:.1f})")
    print(f"  Jacobian eigenvalues: {eigenvalues}")
    print(f"  Classification: {stability}")

In [ ]:
# Phase portrait of the predator-prey system with equilibria marked
fig, ax = plt.subplots(figsize=(9, 7))

# Trajectories from several initial conditions
ics = [[80, 10], [20, 40], [60, 5], [10, 10], [90, 30], [40, 50]]
t_eval_pp = np.linspace(0, 80, 2000)
for ic in ics:
    sol = solve_ivp(predator_prey, (0, 80), ic, t_eval=t_eval_pp, max_step=0.1)
    ax.plot(sol.y[0], sol.y[1], linewidth=1.2, alpha=0.8)
    ax.plot(ic[0], ic[1], 'ko', markersize=4)

# Mark equilibria
markers = {'Stable spiral': ('g*', 15), 'Saddle': ('rx', 12), 'Unstable': ('r^', 10)}
ax.plot(0, 0, 'rx', markersize=12, markeredgewidth=2, label='(0,0) — unstable')
ax.plot(K_pp, 0, 'rx', markersize=12, markeredgewidth=2, label=f'({K_pp:.0f},0) — saddle')
ax.plot(x_star, y_star, 'g*', markersize=15,
        label=f'({x_star:.0f},{y_star:.0f}) — stable spiral')

ax.set_xlabel('Prey $x$'); ax.set_ylabel('Predator $y$')
ax.set_title('Phase portrait with classified equilibria')
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)
ax.set_xlim(0, 110); ax.set_ylim(0, 60)
plt.tight_layout()
plt.show()
# Figure: Trajectories spiral into the coexistence equilibrium.

## 3. Classification of Equilibria (Classification des points d'équilibre)

For a 2D linear system $\mathbf{u}' = A\mathbf{u}$, let $\tau = \text{tr}(A)$ and $\delta = \det(A)$, with eigenvalues $\lambda_{1,2} = \frac{\tau \pm \sqrt{\tau^2 - 4\delta}}{2}$.

| Region | Type |
|:---|:---|
| $\delta < 0$ | Saddle (col) |
| $\delta > 0$, $\tau^2 - 4\delta > 0$, $\tau < 0$ | Stable node (noeud stable) |
| $\delta > 0$, $\tau^2 - 4\delta > 0$, $\tau > 0$ | Unstable node (noeud instable) |
| $\delta > 0$, $\tau^2 - 4\delta < 0$, $\tau < 0$ | Stable spiral (foyer stable) |
| $\delta > 0$, $\tau^2 - 4\delta < 0$, $\tau > 0$ | Unstable spiral (foyer instable) |
| $\delta > 0$, $\tau = 0$ | Center (centre) |

In [ ]:
# Trace-Determinant classification diagram
fig, ax = plt.subplots(figsize=(9, 7))

tau_range = np.linspace(-4, 4, 500)
delta_range = np.linspace(-3, 5, 500)
TAU, DELTA = np.meshgrid(tau_range, delta_range)

# Parabola: tau^2 = 4*delta  =>  delta = tau^2/4
parabola = tau_range**2 / 4

# Regions
# delta < 0: saddle
ax.fill_between(tau_range, -3, 0, alpha=0.15, color='red', label='Saddle')
# delta > 0, above parabola, tau < 0: stable node
# delta > 0, above parabola, tau > 0: unstable node
# delta > 0, below parabola, tau < 0: stable spiral
# delta > 0, below parabola, tau > 0: unstable spiral

ax.fill_between(tau_range[tau_range < 0],
                np.maximum(parabola[tau_range < 0], 0), 5,
                alpha=0.15, color='blue', label='Stable node')
ax.fill_between(tau_range[tau_range > 0],
                np.maximum(parabola[tau_range > 0], 0), 5,
                alpha=0.15, color='orange', label='Unstable node')
ax.fill_between(tau_range[tau_range < 0], 0,
                np.minimum(parabola[tau_range < 0], 5),
                alpha=0.15, color='green', label='Stable spiral')
ax.fill_between(tau_range[tau_range > 0], 0,
                np.minimum(parabola[tau_range > 0], 5),
                alpha=0.15, color='purple', label='Unstable spiral')

ax.plot(tau_range, parabola, 'k-', linewidth=2, label='$\\tau^2 = 4\\delta$ (repeated roots)')
ax.axhline(y=0, color='k', linewidth=1)
ax.axvline(x=0, color='gray', linewidth=0.5, linestyle='--')
ax.plot(0, 0, 'ko', markersize=8)

ax.set_xlabel('Trace $\\tau = \\text{tr}(A)$', fontsize=13)
ax.set_ylabel('Determinant $\\delta = \\det(A)$', fontsize=13)
ax.set_title('Trace-Determinant Classification Diagram', fontsize=14)
ax.legend(loc='upper left', fontsize=10)
ax.set_xlim(-4, 4); ax.set_ylim(-3, 5)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
# Figure: The trace-determinant plane partitions all 2D linear systems
# into distinct dynamical types.

## 4. Lyapunov Stability (Stabilité de Lyapunov)

**Lyapunov's direct method** determines stability without solving the ODE.

**Definition.** An equilibrium $\mathbf{x}^*$ is:
- **Stable** (stable) if for every $\varepsilon > 0$, there exists $\delta > 0$ such that $\|\mathbf{x}(0) - \mathbf{x}^*\| < \delta \implies \|\mathbf{x}(t) - \mathbf{x}^*\| < \varepsilon$ for all $t \ge 0$.
- **Asymptotically stable** (asymptotiquement stable) if stable and $\mathbf{x}(t) \to \mathbf{x}^*$ as $t \to \infty$.

**Lyapunov's theorem.** If there exists a continuously differentiable function $V(\mathbf{x})$ such that:
1. $V(\mathbf{x}^*) = 0$ and $V(\mathbf{x}) > 0$ for $\mathbf{x} \ne \mathbf{x}^*$ (positive definite)
2. $\dot{V} = \nabla V \cdot \mathbf{F} \le 0$ (non-increasing along trajectories)

Then $\mathbf{x}^*$ is **stable**. If additionally $\dot{V} < 0$ for $\mathbf{x} \ne \mathbf{x}^*$, then it is **asymptotically stable**.

### Example: Damped pendulum

$\theta'' + b\theta' + \sin\theta = 0$ with Lyapunov function $V = \frac{1}{2}\dot{\theta}^2 + (1 - \cos\theta)$ (total energy).

In [ ]:
# Damped pendulum: theta'' + b*theta' + sin(theta) = 0
b_damp = 0.3

def pendulum(t, state):
    theta, omega = state
    return [omega, -b_damp * omega - np.sin(theta)]

# Lyapunov function: V = 0.5*omega^2 + (1 - cos(theta))
def lyapunov_V(theta, omega):
    return 0.5 * omega**2 + (1 - np.cos(theta))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Phase portrait
t_eval_pend = np.linspace(0, 40, 2000)
ics_pend = [[0.5, 0], [2.0, 0], [3.0, 0], [0, 2], [-2, 1], [2.5, -1]]
for ic in ics_pend:
    sol = solve_ivp(pendulum, (0, 40), ic, t_eval=t_eval_pend, max_step=0.05)
    axes[0].plot(sol.y[0], sol.y[1], linewidth=1.2)

axes[0].plot(0, 0, 'g*', markersize=12, label='Stable eq.')
axes[0].set_xlabel('$\\theta$'); axes[0].set_ylabel('$\\dot{\\theta}$')
axes[0].set_title('Phase portrait'); axes[0].grid(True, alpha=0.3)
axes[0].legend()

# V(t) along trajectories — should decrease
for ic in ics_pend[:3]:
    sol = solve_ivp(pendulum, (0, 40), ic, t_eval=t_eval_pend, max_step=0.05)
    V_vals = lyapunov_V(sol.y[0], sol.y[1])
    axes[1].plot(sol.t, V_vals, linewidth=1.5, label=f'$\\theta_0={ic[0]}$')

axes[1].set_xlabel('Time $t$'); axes[1].set_ylabel('$V(\\theta, \\dot{\\theta})$')
axes[1].set_title('Lyapunov function $V$ decreases along trajectories')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

# Contours of V
th = np.linspace(-4, 4, 200)
om = np.linspace(-3, 3, 200)
TH, OM = np.meshgrid(th, om)
V_grid = lyapunov_V(TH, OM)
cs = axes[2].contour(TH, OM, V_grid, levels=15, cmap='viridis')
axes[2].clabel(cs, fontsize=8)
axes[2].plot(0, 0, 'r*', markersize=12)
axes[2].set_xlabel('$\\theta$'); axes[2].set_ylabel('$\\dot{\\theta}$')
axes[2].set_title('Level curves of $V$ (Lyapunov function)')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
# Figure: Left — spiraling trajectories. Center — V is monotonically decreasing.
# Right — V level curves are concentric around the equilibrium.

## 5. Bifurcation: Logistic Equation with Harvesting

Returning to $\frac{dN}{dt} = rN(1 - N/K) - H$, as we increase the harvest rate $H$, the two equilibria approach each other and **collide** at the critical value $H_c = rK/4$ (a **saddle-node bifurcation** / bifurcation noeud-col). For $H > H_c$, no equilibrium exists and the population collapses.

A **bifurcation diagram** (diagramme de bifurcation) plots the equilibria $N^*$ as a function of the parameter $H$.

In [ ]:
# Bifurcation diagram: logistic with harvesting
r_bif, K_bif = 1.0, 100.0
H_crit = r_bif * K_bif / 4  # = 25

H_range = np.linspace(0, 30, 500)
N_star_upper = np.full_like(H_range, np.nan)
N_star_lower = np.full_like(H_range, np.nan)

for i, H in enumerate(H_range):
    disc = K_bif**2 - 4*K_bif*H/r_bif
    if disc >= 0:
        N_star_upper[i] = (K_bif + np.sqrt(disc)) / 2
        N_star_lower[i] = (K_bif - np.sqrt(disc)) / 2

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Bifurcation diagram
ax1.plot(H_range, N_star_upper, 'b-', linewidth=2.5, label='Stable equilibrium')
ax1.plot(H_range, N_star_lower, 'r--', linewidth=2.5, label='Unstable equilibrium')
ax1.axvline(x=H_crit, color='gray', linestyle=':', linewidth=1.5,
            label=f'$H_c = rK/4 = {H_crit}$')
ax1.plot(H_crit, K_bif/2, 'ko', markersize=10, label='Saddle-node bifurcation')
ax1.fill_between(H_range[H_range > H_crit], 0, 110, alpha=0.1, color='red')
ax1.text(27, 80, 'Population\ncollapse', fontsize=11, color='red', ha='center')
ax1.set_xlabel('Harvest rate $H$', fontsize=12)
ax1.set_ylabel('Equilibrium $N^*$', fontsize=12)
ax1.set_title('Saddle-Node Bifurcation Diagram\n(Bifurcation noeud-col)', fontsize=13)
ax1.legend(fontsize=10); ax1.grid(True, alpha=0.3)
ax1.set_ylim(0, 110)

# Time evolution near bifurcation
for H_val, style in [(20, '-'), (24, '--'), (25, ':'), (26, '-')]:
    sol = solve_ivp(logistic_harvest, (0, 40), [60.0], args=(H_val,),
                    t_eval=np.linspace(0, 40, 500))
    ax2.plot(sol.t, sol.y[0], style, linewidth=2, label=f'H={H_val}')

ax2.axhline(y=0, color='k', linewidth=0.5)
ax2.set_xlabel('Time'); ax2.set_ylabel('Population $N(t)$')
ax2.set_title('Dynamics near the bifurcation point')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
# Figure: Left — bifurcation diagram showing equilibria merging at H_c.
# Right — near-critical harvesting leads to delayed collapse (critical slowing down).

## 6. Phase Portrait Gallery with Classification

We create a systematic gallery showing all major equilibrium types with their eigenvalue signatures, applied to concrete nonlinear systems linearised around their equilibria.

In [ ]:
def plot_classified_portrait(f, xlim, ylim, title, eq_points, ax):
    """Plot phase portrait with classified equilibria."""
    x = np.linspace(*xlim, 20)
    y = np.linspace(*ylim, 20)
    X, Y = np.meshgrid(x, y)
    U = np.zeros_like(X)
    V = np.zeros_like(Y)
    for i in range(X.shape[0]):
        for j in range(X.shape[1]):
            res = f(0, [X[i,j], Y[i,j]])
            U[i,j], V[i,j] = res[0], res[1]
    
    ax.streamplot(X, Y, U, V, density=1.2, color='steelblue',
                  linewidth=0.7, arrowsize=1)
    
    for (ex, ey), label, color, marker in eq_points:
        ax.plot(ex, ey, marker, color=color, markersize=10, markeredgewidth=2)
        ax.annotate(label, (ex, ey), textcoords="offset points",
                    xytext=(10, 10), fontsize=8)
    
    ax.set_xlabel('$x$'); ax.set_ylabel('$y$')
    ax.set_title(title, fontsize=11)
    ax.grid(True, alpha=0.3)

fig, axes = plt.subplots(2, 2, figsize=(13, 12))

# 1. Van der Pol oscillator (limit cycle)
mu_vdp = 1.0
f_vdp = lambda t, z: [z[1], mu_vdp*(1 - z[0]**2)*z[1] - z[0]]
plot_classified_portrait(f_vdp, (-4, 4), (-4, 4),
    'Van der Pol: unstable spiral + limit cycle',
    [((0, 0), 'Unstable spiral', 'red', 'o')], axes[0, 0])

# 2. Saddle-node system: x' = x^2, y' = -y
f_sn = lambda t, z: [z[0]**2 - 1, -z[1]]
plot_classified_portrait(f_sn, (-2.5, 2.5), (-2, 2),
    "$x'=x^2-1, y'=-y$",
    [((-1, 0), 'Stable node', 'green', '*'),
     ((1, 0), 'Saddle', 'red', 'x')], axes[0, 1])

# 3. Nonlinear center: x' = y, y' = -sin(x)
f_pend = lambda t, z: [z[1], -np.sin(z[0])]
plot_classified_portrait(f_pend, (-5, 5), (-3, 3),
    'Undamped pendulum: centers + saddles',
    [((0, 0), 'Center', 'blue', 'o'),
     ((np.pi, 0), 'Saddle', 'red', 'x'),
     ((-np.pi, 0), 'Saddle', 'red', 'x')], axes[1, 0])

# 4. Two competing attractors
f_bist = lambda t, z: [z[0] - z[0]**3 - z[0]*z[1]**2,
                        z[1] - z[1]**3 - z[0]**2*z[1]]
plot_classified_portrait(f_bist, (-1.5, 1.5), (-1.5, 1.5),
    'Gradient system with multiple equilibria',
    [((0, 0), 'Unstable', 'red', 'o'),
     ((1, 0), 'Stable', 'green', '*'),
     ((-1, 0), 'Stable', 'green', '*'),
     ((0, 1), 'Stable', 'green', '*'),
     ((0, -1), 'Stable', 'green', '*')], axes[1, 1])

plt.tight_layout()
plt.show()
# Figure: Gallery of nonlinear systems exhibiting different equilibrium types.

## 7. Exercise: Stability Analysis of an Epidemic Model

Consider the **SIS model** (without immunity — recovered individuals become susceptible again):

$$\frac{dI}{dt} = \beta(N - I)I - \gamma I = (\beta N - \gamma)I - \beta I^2$$

This is a single ODE with two equilibria:
- **Disease-free equilibrium** (DFE): $I^* = 0$
- **Endemic equilibrium** (EE): $I^* = N - \gamma/\beta = N(1 - 1/R_0)$ where $R_0 = \beta N/\gamma$

**Tasks:**
1. Compute $f'(I^*)$ at each equilibrium. Show that:
   - DFE is stable when $R_0 < 1$ and unstable when $R_0 > 1$
   - EE is stable when $R_0 > 1$ (and biologically meaningful only then)
2. This is a **transcritical bifurcation** at $R_0 = 1$. Draw the bifurcation diagram: plot $I^*$ vs $R_0$.
3. Verify numerically: solve the ODE for $R_0 = 0.8, 1.0, 1.5, 3.0$ with $N = 1000$.
4. **Extension to SIR:** For the full SIR model with disease-free equilibrium $(S^*, 0, R^*)$, compute the Jacobian and show that stability depends on $\beta S^*/\gamma$.

In [ ]:
# SIS model exercise — starter code and visualisation
N_sis = 1000
gamma_sis = 0.1

def sis_model(t, I, beta):
    return beta * (N_sis - I) * I - gamma_sis * I

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Time evolution for different R_0
t_eval_sis = np.linspace(0, 100, 500)
for R0_val in [0.5, 0.8, 1.0, 1.5, 3.0]:
    beta_val = R0_val * gamma_sis / N_sis
    sol = solve_ivp(sis_model, (0, 100), [10.0], args=(beta_val,), t_eval=t_eval_sis)
    label = f'$R_0={R0_val}$'
    if R0_val > 1:
        I_ee = N_sis * (1 - 1/R0_val)
        label += f' ($I^*={I_ee:.0f}$)'
    ax1.plot(sol.t, sol.y[0], linewidth=2, label=label)

ax1.set_xlabel('Time'); ax1.set_ylabel('Infected $I(t)$')
ax1.set_title('SIS model: convergence to equilibrium')
ax1.legend(fontsize=9); ax1.grid(True, alpha=0.3)

# Bifurcation diagram
R0_range = np.linspace(0, 4, 300)
I_dfe = np.zeros_like(R0_range)  # disease-free: always exists
I_ee = np.where(R0_range > 1, N_sis * (1 - 1/R0_range), np.nan)

# Stability: DFE stable for R0<1, EE stable for R0>1
ax2.plot(R0_range[R0_range <= 1], I_dfe[R0_range <= 1], 'b-', linewidth=3, label='DFE (stable)')
ax2.plot(R0_range[R0_range > 1], I_dfe[R0_range > 1], 'b--', linewidth=2, label='DFE (unstable)')
ax2.plot(R0_range[R0_range > 1], I_ee[R0_range > 1], 'r-', linewidth=3, label='Endemic eq. (stable)')
ax2.axvline(x=1, color='gray', linestyle=':', label='$R_0 = 1$ (transcritical bifurcation)')

ax2.set_xlabel('Basic reproduction number $R_0$', fontsize=12)
ax2.set_ylabel('Equilibrium infected $I^*$', fontsize=12)
ax2.set_title('Transcritical Bifurcation in SIS Model\n(Bifurcation transcritique)', fontsize=13)
ax2.legend(fontsize=9); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
# Figure: Left — I(t) converges to DFE when R_0<1, to endemic equilibrium when R_0>1.
# Right — transcritical bifurcation diagram: exchange of stability at R_0=1.